# Example: The Pricing of United States Treasury Coupon Bearing Notes and Bonds
In this example, we price United States Treasury notes and bonds by discounting their coupon and principal cash flows to the auction date.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Construct coupon-security cash flows:__ Represent the purchase price, periodic coupons, and final principal payment at their correct dates.
> * __Price notes and bonds using NPV:__ Discount each promised payment and solve the zero-NPV condition for the security's price.
> * __Compare model and auction prices:__ Apply the valuation model to Treasury data and interpret differences in light of timing and quotation conventions.

We will construct the cash flows and compare the calculated prices with auction observations. Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

Let's set up our code environment:

In [1]:
# Load the shared L2a paths, course environment, and package imports.
include(joinpath(@__DIR__, "Include.jl"));

  Activating project at `~/Desktop/julia_work/CHEME-5660-CourseRepository-Fall-2026`


For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 Quantitative Finance Package documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

### Data
We'll explore T-note and T-bond prices from United States Treasury auctions since October 2022, downloaded using the [Auction query functionality of TreasuryDirect.gov](https://www.treasurydirect.gov/auctions/auction-query/) and vendored with the course package. 

We load the dataset using [the `MyTreasuryNotesAndBondsDataSet(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/data/#VLQuantitativeFinancePackage.MyTreasuryNotesAndBondsDataSet) exported by the `VLQuantitativeFinancePackage`, and store the auction records in the `dataset::DataFrame` variable. By default it returns original issues only; reopenings carry an existing security's coupon and so trade well away from par, and are available with `reopenings = true`.

The dataset also includes inflation-protected securities (TIPS), which appear as ordinary `Note` and `Bond` rows but whose `High Yield` is a real yield rather than a nominal one. Pricing inflation-indexed cash flows is out of scope for this example, so we drop those records. In this auction window every real yield sits below 3% and every nominal yield sits above it, which gives us a clean filter.

In [2]:
# Load the Treasury note and bond auction records used in this example.
all_securities = MyTreasuryNotesAndBondsDataSet();

# Retain nominal coupon securities; the TIPS real yields in this sample are below 3%.
dataset = filter(row -> row[Symbol("High Yield")] >= 0.03, all_securities)

Row,CUSIP,Security Type,Security Term,Auction Date,Issue Date,Maturity Date,Price,High Yield,Interest Rate
,String15,String7,String31,String15,String15,String15,Float64,Float64,Float64
1,912810UX4,Bond,20-Year,08/19/2026,08/31/2026,08/15/2046,99.0213,0.05204,0.05125
2,912810UW6,Bond,30-Year,08/13/2026,08/17/2026,08/15/2056,98.627,0.05216,0.05125
3,91282CRF0,Note,10-Year,08/12/2026,08/17/2026,08/15/2036,99.5407,0.04683,0.04625
4,91282CRG8,Note,3-Year,08/11/2026,08/17/2026,08/15/2029,99.8854,0.04291,0.0425
5,91282CRC7,Note,7-Year,07/28/2026,07/31/2026,07/31/2033,99.4165,0.04473,0.04375
6,91282CRB9,Note,2-Year,07/27/2026,07/31/2026,07/31/2028,99.8767,0.04315,0.0425
7,91282CRA1,Note,5-Year,07/27/2026,07/31/2026,07/31/2031,99.8534,0.04408,0.04375
8,91282CQZ7,Note,3-Year,07/07/2026,07/15/2026,07/15/2029,99.8492,0.04179,0.04125
9,91282CQW4,Note,7-Year,06/25/2026,06/30/2026,06/30/2033,99.94,0.0426,0.0425


Let's store the dimension (number of records) of our treasury auction dataset in the `number_of_records::Int64` variable using [the `nrow(...)` function exported by the DataFrames.jl package](https://dataframes.juliadata.org/stable/lib/functions/#DataAPI.nrow)

In [3]:
# Count the auction records that will be repriced in Task 2.
number_of_records = nrow(dataset)

233

Before we dive into pricing calculations, think about what factors might influence the price of a Treasury security. What role do you think the coupon rate, time to maturity, and prevailing interest rates play in determining a bond's price?

___

## Task 1: Visualize the cash flows and discounting for coupon-bearing Treasury bonds
Unlike zero-coupon Treasury bills, which have only two cash flow events (investors provide funds to the Treasury and receive the face (par) value at maturity), coupon-bearing Treasury securities are more complicated because of the periodic coupon payments. Thus, it's helpful to visualize the cash flow events of notes and bonds. 

We begin by building [an instance of the `DiscreteCompoundingModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.DiscreteCompoundingModel) and store this discount model in the `discount_model::DiscreteCompoundingModel` variable:

In [4]:
# Select discrete compounding for all note and bond valuations below.
discount_model = DiscreteCompoundingModel();

Next, let's build an instance of [the `MyUSTreasuryCouponSecurityModel` type](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel) using [the `build(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.build-Tuple{Type{MyUSTreasuryCouponSecurityModel},%20NamedTuple}). 

> __Example:__ We'll compute the price and cash flow for a `T = 20-yr` bond with coupon rate $c=1.750\%$, nominal annual yield $y=1.850\%$, coupon frequency $n=2~\mathrm{yr}^{-1}$, and face (par) value $V_P=100~\mathrm{USD}$. The package API stores these inputs in the fields `coupon = c`, `rate = y`, and `λ = n`. The price reported on [TreasuryDirect.gov](https://www.treasurydirect.gov/marketable-securities/understanding-pricing/#id-for-more-detailed-formulas-and-useful-tables-264977) for this bond is $V_B=98.336~\mathrm{USD}$.

Similar to zero-coupon T-bills, we'll use [the `short-cut` syntax](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#Short-cut-syntax), which relies on the [Julia pipe `|>` operator](https://docs.julialang.org/en/v1/manual/functions/#Function-composition-and-piping) to compute the coupon-bearing price, forward accumulation factors, and discounted cash flows. 

Let's store the result in the `test_bond::MyUSTreasuryCouponSecurityModel` variable. 

In [5]:
# Define the security and market inputs. Rates are stored as decimal fractions.
T = 20.0;     # years remaining to maturity
y = 0.01850;  # nominal annual yield used to value the bond
c = 0.01750;  # annual coupon rate paid by the bond
n = 2;        # coupon payments and yield-compounding periods per year
par = 100.0;  # face value repaid at maturity, in USD

# Build the bond and apply the discrete-compounding pricing model.
test_bond = build(MyUSTreasuryCouponSecurityModel, (
    T = T, rate = y, coupon = c, λ = n, par = par
)) |> discount_model;

Now that we have populated [the `MyUSTreasuryCouponSecurityModel` instance](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/fixed/#VLQuantitativeFinancePackage.MyUSTreasuryCouponSecurityModel) stored in the `test_bond` variable, we can pull data from `test_bond` and construct a table. 

The package stores the price in `test_bond.price`, the discounted cash flows in `test_bond.cashflow`, and the forward accumulation factors in `test_bond.discount`. We assign these values clear local names before constructing the table.

In [6]:
# Extract the computed values needed for the price check and cash-flow table.
nominal_computed_price = trunc(test_bond.price, digits = 3); # price per 100 USD of par
cashflow = test_bond.cashflow;       # time-0 value of each signed cash flow
accumulation = test_bond.discount;   # package field stores forward accumulation factors

# Report the model price using Treasury's three-decimal price convention.
println("This computed bond price = $(nominal_computed_price) USD")

This computed bond price = 98.334 USD


#### Check: Are the computed and observed bond prices similar?
Let's use the [isapprox function](https://docs.julialang.org/en/v1/base/math/#Base.isapprox) combined with the [@assert macro](https://docs.julialang.org/en/v1/base/base/#Base.@assert) to check the similarity of the computed, and observed bond price. 

> __Test:__ If the price values are different beyond a relative tolerance of `rtol = 1e-4`, a `false` result is generated, and an [AssertionError](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) is thrown. Otherwise, nothing happens.

So how did we do?

In [7]:
# Compare the model price with the TreasuryDirect value quoted above.
observed_bond_price = 98.336; # published price per 100 USD of par
@assert isapprox(observed_bond_price, nominal_computed_price; rtol = 1e-4)

The table shows the nominal and discounted cash flows in each period. The model supplies the accumulation factor. We take its reciprocal to obtain the discount factor, then multiply the nominal cash flow by the discount factor. The final column reports the running total of the discounted cash flows.

In [8]:
let
    # Initialize the table dimensions and the bond's contractual payments.
    number_of_periods = length(accumulation); # includes the purchase date at index 0
    final_period = number_of_periods - 1;      # financial index of the maturity payment
    coupon_payment = (test_bond.coupon / test_bond.λ) * test_bond.par; # USD per coupon period
    bond_data_table = Array{Any,2}(undef, number_of_periods, 5); # one row per financial index
    cumulative_discounted_cashflow = 0.0; # running time-0 value, in USD

    # Construct and discount the signed cash flow at each financial index.
    for i ∈ 0:final_period

        accumulation_factor = accumulation[i];     # grows time-0 dollars to period-i dollars
        discount_factor = 1 / accumulation_factor; # returns period-i dollars to time 0

        # The buyer pays the price at i = 0, receives coupons afterward, and
        # receives both the final coupon and par value at maturity.
        nominal_cashflow = if i == 0
            -test_bond.price
        elseif i == final_period
            coupon_payment + test_bond.par
        else
            coupon_payment
        end
        discounted_cashflow = discount_factor * nominal_cashflow; # period-i value in time-0 USD
        cumulative_discounted_cashflow += discounted_cashflow;    # include the current row

        # Verify that the explicit calculation matches the package's stored value.
        @assert isapprox(discounted_cashflow, cashflow[i]; atol = 1e-12)

        # Store the completed row. Julia row i + 1 represents financial index i.
        bond_data_table[i+1,1] = i;
        bond_data_table[i+1,2] = accumulation_factor;
        bond_data_table[i+1,3] = nominal_cashflow;
        bond_data_table[i+1,4] = discounted_cashflow;
        bond_data_table[i+1,5] = cumulative_discounted_cashflow;
    end

    # Display every period with a simple text border and no vertical cropping.
    pretty_table(bond_data_table; 
        column_labels=["Period", "Accumulation factor", "Nominal cashflow", "Discounted cashflow", "Cumulative discounted cashflow"], 
        fit_table_in_display_horizontally = false, fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__simple))
end

========= ===================== ================== ===================== =================================
  Period   Accumulation factor   Nominal cashflow   Discounted cashflow   Cumulative discounted cashflow 
========= ===================== ================== ===================== =================================
       0                   1.0           -98.3346              -98.3346                         -98.3346
       1               1.00925              0.875               0.86698                         -97.4677
       2               1.01859              0.875              0.859034                         -96.6086
       3               1.02801              0.875              0.851161                         -95.7575
       4               1.03752              0.875               0.84336                         -94.9141
       5               1.04711              0.875               0.83563                         -94.0785
       6                1.0568              0.875 

___

## Task 2: Compute the price of notes and bonds observed at auction
Next, we compute the price of Treasury notes and bonds from the `dataset::DataFrame` and compare the estimated cost with the price observed at the auction. We process each entry in the `dataset` using [a `for-loop`](https://docs.julialang.org/en/v1/base/base/#for). 

During each iteration of the loop:
> We get data from the `dataset::DataFrame` and update the model instance. We set the maturity $T$ (converted to years using `securityterm`), nominal annual yield $y$, annual coupon rate $c$, and coupon frequency $n=2~\mathrm{yr}^{-1}$. The package fields are `rate = y`, `coupon = c`, and `λ = n`. We then compute the note or bond price using the short-cut syntax and the discount model. 

We calculate the percentage error between the estimated and observed price and store the data for each iteration in the `computed_price_table::DataFrame` using [the `push!(...)` function](https://dataframes.juliadata.org/stable/lib/functions/#Base.push!). 

Lastly, we store the populated model instance in the `security_dictionary::Dict{Int64, MyUSTreasuryCouponSecurityModel}` dictionary where keys are the index `i,` values are the populated model instances.

In [9]:
security_dictionary, computed_price_table = let
    # Initialize one table row and one priced model for each auction record.
    computed_price_table = DataFrame(); # observed prices, model prices, and errors
    security_dictionary = Dict{Int64,MyUSTreasuryCouponSecurityModel}(); # row index => model
    n = 2; # Treasury notes and bonds pay coupons semiannually

    # Reprice every nominal coupon security in the filtered dataset.
    for i ∈ 1:number_of_records

        # Read the contract terms and auction yield from row i.
        term_label = String(dataset[i, Symbol("Security Term")]); # e.g., "10-Year"
        T = securityterm(term_label); # convert the label to maturity in years
        y = dataset[i, Symbol("High Yield")]; # auction yield as a decimal fraction
        c = dataset[i, Symbol("Interest Rate")]; # annual coupon rate as a decimal fraction

        # Build and price a 100 USD par security using semiannual compounding.
        model = build(MyUSTreasuryCouponSecurityModel, (
            par = 100.0, T = T, rate = y, coupon = c, λ = n
        )) |> discount_model;
        
        # Measure the absolute relative difference from the published auction price.
        auction_price = dataset[i, :Price]; # USD per 100 USD of par
        computed_price = model.price;        # USD per 100 USD of par
        relative_error = abs((auction_price - computed_price) / computed_price);

        # Assemble a reader-facing row; rates are converted from decimals to percent.
        results_tuple = (
            CUSIP = dataset[i, :CUSIP],
            term = dataset[i, Symbol("Security Term")],
            rate = y*100,
            coupon = c*100,
            computed = trunc(computed_price, digits = 3),
            actual = trunc(auction_price, digits = 3),
            relative_error = relative_error
        );

        # Store the summary row and its full priced model under the same row index.
        push!(computed_price_table, results_tuple);
        security_dictionary[i] = model;
    end

    # Display all repricing results without horizontal or vertical cropping.
    pretty_table(computed_price_table, 
        fit_table_in_display_horizontally = false, fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__simple)
    );
    security_dictionary, computed_price_table # return both objects from the `let` block
end;

============ ========== ========= ========= ========== ========= =================
      CUSIP       term      rate    coupon   computed    actual   relative_error 
   String15   String31   Float64   Float64    Float64   Float64          Float64 
============ ========== ========= ========= ========== ========= =================
  912810UX4    20-Year     5.204     5.125     99.025    99.021       4.00937e-5
  912810UW6    30-Year     5.216     5.125     98.627    98.627       6.05768e-6
  91282CRF0    10-Year     4.683     4.625     99.541     99.54       3.77033e-6
  91282CRG8     3-Year     4.291      4.25     99.885    99.885        2.8872e-6
  91282CRC7     7-Year     4.473     4.375     99.416    99.416       1.82182e-9
  91282CRB9     2-Year     4.315      4.25     99.876    99.876       3.54689e-9
  91282CRA1     5-Year     4.408     4.375     99.853    99.853      9.43135e-10
  91282CQZ7     3-Year     4.179     4.125     99.849    99.849       1.11091e-9
  91282CQW4     7-Year

### Check: How well do we estimate the price of notes and bonds at auction?
Let's specify a tolerance and compute the fraction of notes and bonds with a relative error _less than or equal_ to the tolerance.
* Let the `tolerance = 1e-3`. You can specify a different value by setting the `tolerance` variable
* We iterate through the entries of the `computed_price_table,` and increment the `counter` variable if the `relative_error` $\leq$ `tolerance.`
* Finally, we compute the fraction of notes and bonds that satisfy the relative error check

Let's see how many instruments clear the tolerance.

In [10]:
# Set the largest acceptable relative pricing error: 10⁻³ = 0.1%.
tolerance = 1e-3;
counter = 0; # number of auction records that satisfy the tolerance

# Count records whose model price is sufficiently close to the auction price.
for i ∈ 1:number_of_records
    if computed_price_table[i, :relative_error] ≤ tolerance
        counter += 1;
    end
end

# Convert the count to the fraction of all repriced securities.
fraction = counter / number_of_records;
println("What fraction of instruments satisfy the error tolerance: $(fraction) \
    or $(counter) out of $(number_of_records)");

What fraction of instruments satisfy the error tolerance: 1.0 or 233 out of 233


### Discussion question
1. Based on the fraction of instruments that meet our error tolerance, what can you conclude about the effectiveness of the NPV pricing model for Treasury securities? What factors might explain any pricing discrepancies you observe?
2. If `fraction` does not equal `1.0`, which note or bond was mispriced? Did the buyer of the mispriced note or bond get a `good` or `bad` deal, i.e., did they underpay or overpay?
3. What other factors should investors consider when evaluating Treasury securities? How might liquidity, credit risk, and inflation expectations influence investment decisions?

___

## Summary

This example priced coupon-bearing Treasury notes and bonds by placing every promised cash flow on its payment date and discounting those cash flows to the auction date.

> __Key Takeaways:__
>
> * __Cash-flow timing determines value:__ A coupon security combines periodic coupon payments with the repayment of principal at maturity, and each payment must be discounted over its own time interval.
> * __Price follows from the zero-NPV condition:__ The model price is the purchase-date amount that makes the net present value of the purchase outflow and promised inflows equal to zero under the stated yield convention.
> * __Auction comparisons require consistent conventions:__ Differences between calculated and reported prices can reflect accrued interest, settlement timing, day-count rules, quotations, rounding, or an incorrect model input.

The NPV framework provides a reproducible valuation benchmark, but its comparison with a market quotation is meaningful only when the cash flows, dates, and quotation conventions are aligned.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance.  Only risk capital that is not required for living expenses.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.